# Pipeline

In [43]:
import networkx as nx
import matplotlib.pyplot as plt
import random
from sklearn.metrics import roc_auc_score, roc_curve

## Parameters

In [44]:
network = nx.watts_strogatz_graph(300, 2, 0.3) # Network
beta = 0.3 # Infection rate
gamma = 0.28 # Recovery rate
initial_infected_count = 1 # Number of initial infected nodes
asymptomatic_rate = 0.25 # Fraction of asymptomatic nodes

## Defining asymptomatics

In [45]:
def choose_asymptomatic_nodes(graph, fraction, seed=None):
    random.seed(seed)
    nodes = list(graph.nodes())
    k = int(len(nodes) * fraction)
    return set(random.sample(nodes, k))

In [46]:
asymptomatic = choose_asymptomatic_nodes(network, fraction = asymptomatic_rate)
len(asymptomatic), asymptomatic

(75,
 {3,
  4,
  5,
  8,
  11,
  12,
  13,
  14,
  21,
  24,
  26,
  29,
  31,
  32,
  34,
  35,
  38,
  44,
  46,
  52,
  54,
  55,
  59,
  60,
  62,
  68,
  72,
  74,
  76,
  77,
  79,
  82,
  86,
  97,
  102,
  106,
  117,
  124,
  125,
  131,
  132,
  135,
  140,
  141,
  150,
  155,
  156,
  162,
  163,
  164,
  174,
  186,
  201,
  202,
  208,
  223,
  233,
  235,
  238,
  243,
  249,
  250,
  255,
  267,
  270,
  272,
  273,
  276,
  280,
  283,
  291,
  292,
  294,
  296,
  297})

## SIS epidemic

In [47]:
def epidemic_step(graph, beta, gamma, infected):

    new_infected = set()
    recovered = set()

    for node in infected:
        for neighbor in set(graph.neighbors(node)) - infected:
            if random.random() < beta:
                new_infected.add(neighbor)

    for node in infected:
        if random.random() < gamma:
            recovered.add(node)

    infected = infected | new_infected
    infected = infected - recovered

    return infected


def burn_in(graph, beta, gamma, initial_infected_count, burn_in_steps,):
    initial_infected = random.sample(list(graph.nodes()), k=initial_infected_count) # Randomly select the initial infected nodes from the graph
    infected = set(initial_infected)

    for _ in range(burn_in_steps):
        infected = epidemic_step(graph, beta, gamma, infected)
    
    return infected

In [48]:
infected = burn_in(network, beta, gamma, initial_infected_count, 1000)
len(infected), infected

(30,
 {6,
  7,
  43,
  44,
  79,
  98,
  99,
  103,
  104,
  106,
  107,
  108,
  112,
  118,
  119,
  132,
  134,
  135,
  184,
  187,
  193,
  194,
  230,
  238,
  239,
  240,
  241,
  247,
  248,
  257})

## Collecting snapshots

In [49]:
def collect_snapshots(graph, beta, gamma, infected, num_snapshots, step_between):
    snapshots = []

    for _ in range(num_snapshots):

        for _ in range(step_between):
            infected = epidemic_step(graph, beta, gamma, infected)

        snapshots.append(set(infected))

    return snapshots

Salva o conjunto de todos os infectados em cada snapshot

In [50]:
snapshots = collect_snapshots(network, beta, gamma, infected, 4, 1000)
for i in snapshots:
    print(len(i))

46
35
25
46


## Observed betweenness

In [51]:
def observed_betweenness(graph, observed_nodes):
    """
    Compute the betweenness centrality of nodes considering only
    shortest paths that have both source and target nodes in the
    observed infected nodes set.

    Parameters:
    - graph: NetworkX graph
    - observed_nodes: List of observed infected nodes

    Returns:
    - A dictionary with the betweenness centrality scores.
    """

    return nx.betweenness_centrality_subset(graph, sources=observed_nodes, targets=observed_nodes, normalized=True)

In [52]:
betws = []
observations = {}

for k, s in enumerate(snapshots):
    obs = s - asymptomatic
    observations[k] = obs
    obs_betw = observed_betweenness(network, obs)
    betws.append(obs_betw)

In [53]:
for i in observations.values():
    print(len(i))

35
27
21
37


In [54]:
betws[3]

{0: 0.0,
 1: 0.0,
 2: 0.0,
 3: 0.0,
 4: 0.0,
 5: 0.0,
 6: 0.0,
 7: 0.0,
 8: 0.0,
 9: 0.0,
 10: 0.0,
 11: 0.0,
 12: 0.0,
 13: 0.0,
 14: 0.0,
 15: 0.0,
 16: 0.0,
 17: 0.0,
 18: 0.0,
 19: 0.0,
 20: 0.0,
 21: 0.0,
 22: 0.0019303719332899375,
 23: 0.008125519068034388,
 24: 0.0,
 25: 0.0,
 26: 0.0,
 27: 0.0,
 28: 0.0,
 29: 0.0,
 30: 0.0,
 31: 0.0,
 32: 0.0,
 33: 0.0,
 34: 0.0,
 35: 0.0,
 36: 0.0,
 37: 0.0,
 38: 0.0,
 39: 0.0,
 40: 0.0,
 41: 0.0,
 42: 0.0,
 43: 0.0,
 44: 0.0,
 45: 0.0,
 46: 0.0,
 47: 0.0,
 48: 0.0,
 49: 0.0,
 50: 0.0,
 51: 0.0,
 52: 0.0,
 53: 0.0,
 54: 0.0,
 55: 0.0,
 56: 0.0,
 57: 0.0,
 58: 0.0,
 59: 0.0,
 60: 0.0,
 61: 0.0,
 62: 0.0,
 63: 0.0,
 64: 0.0,
 65: 0.0,
 66: 0.0,
 67: 0.0,
 68: 0.0,
 69: 0.0,
 70: 0.0,
 71: 0.0,
 72: 0.0,
 73: 0.0,
 74: 0.0,
 75: 0.0,
 76: 0.0008080626697492761,
 77: 0.0008080626697492761,
 78: 0.0,
 79: 0.0,
 80: 0.0,
 81: 0.0,
 82: 0.0,
 83: 0.0,
 84: 0.0,
 85: 0.0,
 86: 0.0,
 87: 0.0,
 88: 0.0,
 89: 0.0,
 90: 0.0,
 91: 0.0,
 92: 0.0,
 93: 0.0,

## Evaluating

In [55]:
def auc_score(infected_nodes, centrality_measures, observed_nodes):
    """
    Calculate the AUC score and ROC curve, given the set of infected nodes,
    the network centrality measure used for ranking, and a list of observed 
    nodes to be excluded from the calculation.

    Parameters:
    - infected_nodes: List of infected nodes
    - centrality_measures: Dictionary where the keys are node indices and
                           the values are their corresponding centrality
                           measures (e.g., degree, betweenness, etc.)
    - observed_nodes : List of node indices to exclude from evaluation
    
    Returns:
    - auc: The Area Under the Curve (AUC) score
    - curve: Tuple containing the False Positive Rate (FPR),
             True Positive Rate (TPR), and thresholds for
             plotting the ROC curve
    """
    # Filter out observed nodes
    evaluation_nodes = [node for node in centrality_measures if node not in observed_nodes]

    # Build ground truth and scores
    y_true = []
    y_scores = []

    for node in evaluation_nodes:
        y_true.append(1 if node in infected_nodes else 0)
        y_scores.append(centrality_measures[node])

    # Compute the AUC score
    auc = roc_auc_score(y_true, y_scores)
    # Compute the ROC curve
    curve = roc_curve(y_true, y_scores)

    return {'auc': auc, 'roc': curve}

In [56]:
AUCS = []

for k, s in enumerate(snapshots):
    print("iteration", k)

    betw = {i: sum(d[i] for d in betws[:k+1]) for i in betws[0]}
    obs = set().union(*(observations[i] for i in range(k+1)))
    AUC = auc_score(s, betw, obs)['auc'] # Check if this makes sense
    AUCS.append(AUC)

print("AUCS:", AUCS)

iteration 0
iteration 1
iteration 2
iteration 3
AUCS: [np.float64(0.8702576950608447), np.float64(0.7663934426229508), np.float64(0.7916666666666666), np.float64(0.9593621399176955)]
